In [0]:

# Databricks notebook source
# /// script
# [tool.databricks.environment]
# environment_version = "6"
# ///
# ==============================================================================
# PIPELINE DEPENDENCY & IMPACT TRACKING
# ==============================================================================
# Upstream Dependencies:
#   - [Module]: notebooks/00_config.py        (PATHS, TABLE_NAMES, DEFAULT_LOOKBACK_DAYS,
#                                              get_pipeline_parameter, configure_spark_runtime)
#   - [Module]: notebooks/00_audit_logger.py  (PipelineAuditor, layer = "Bronze-to-Silver")
#   - [Delta Table]: bronze_ooni_raw            (01_bronze_ingestion.py, Phase 0 contract:
#                                                measurement_uid, api_anomaly, api_confirmed)
#   - [Delta Table]: ref_citizenlab_categories  (01_bronze_ingestion.py, with taxonomy_version)
# Downstream Impact:
#   - [Generates Delta Table]: silver_network_measurements        (1 row per measurement)
#   - [Generates Delta Table]: silver_dns_experiments             (1 row per DNS query)
#   - [Generates Delta Table]: silver_tcp_experiments             (1 row per TCP handshake)
#   - [Generates Delta Table]: silver_app_anomaly_measurements    (messaging / circumvention tests)
#   - [Appends Delta Table]:   quarantine_ooni_raw  (_source_file = 'BRONZE_TO_SILVER')
#   - [Appends Delta Table]:   pipeline_execution_logs (layer = 'Bronze-to-Silver')
#   - [Schema Contracts]:      data/schemas/silver_*_schema.json (must match the *_SCHEMA objects below)
#   - [Affects]: Gold CAI / dashboards that read the tables above
# Risk & Side Effects:
#   - Changing the measurement_id formula, the PII pepper, or any *_SCHEMA requires a full Silver rebuild.
#   - Changing classification / taxonomy rules is SAFE: the parent MERGE updates affected rows in place.
#   - PII guardrail: probe_id, probe_ip, raw test_keys, annotations, test_helpers, extensions and the raw
#     resolver IP / resolver_address never reach any Silver table, log line or quarantine record.
# ==============================================================================
 
"""
Project Panopticon: Silver Transformation Engine (Bronze -> Silver), phases 1-6.
 
  Phase 1  canonical measurement_id, privacy boundary, fail-closed pepper, ASN fallbacks
  Phase 2  two-tier Citizen Lab enrichment (exact URL -> domain fallback -> unmatched)
  Phase 3  epistemic classification (evidence flags decoupled from confidence status)
  Phase 4  independent posexplode_outer child projections + polymorphic app table
  Phase 5  partition-pruned, zero-write-amplification Delta MERGE
  Phase 6  schema-contract validator (names AND types)
"""
 
import importlib
import hashlib
import hmac
import json
import os
from datetime import date, datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple
 
try:
    from pyspark.sql import SparkSession, DataFrame, Window
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, LongType, DoubleType,
        IntegerType, BooleanType, TimestampType, DateType,
    )
    from delta.tables import DeltaTable
    HAS_PYSPARK = True
except (ImportError, ModuleNotFoundError):
    SparkSession = DataFrame = DeltaTable = Any  # type: ignore
    HAS_PYSPARK = False
 
config = None
for _mod_name in ("notebooks.00_config", "00_config"):
    try:
        config = importlib.import_module(_mod_name)
        break
    except (ImportError, ModuleNotFoundError):
        continue
if config is None:
    raise ImportError("[02_silver_transformation] Could not import '00_config'.")
 
audit_logger = None
for _mod_name in ("notebooks.00_audit_logger", "00_audit_logger"):
    try:
        audit_logger = importlib.import_module(_mod_name)
        break
    except (ImportError, ModuleNotFoundError):
        continue
 
 
 

In [0]:
 
# ==============================================================================
# STEP 0: CONTRACTS (explicit schemas; inferSchema is never used)
# ==============================================================================
 
def _f(name: str, dtype: Any, nullable: bool, comment: str) -> "StructField":
    return StructField(name, dtype, nullable, {"comment": comment})
 
 
# --- 0.1 Parent table: one row per measurement ---------------------------------
SILVER_SCHEMA = StructType([
    _f("measurement_id",           StringType(),    False, "PK: OONI measurement_uid when present, else length-prefixed SHA-256 of the 6 Bronze natural-key parts"),
    _f("report_id",                StringType(),    False, "Provenance key back to the OONI report"),
    _f("event_timestamp",          TimestampType(), False, "Measurement start time, UTC"),
    _f("measurement_date",         DateType(),      False, "Partition column (UTC date of event_timestamp)"),
    _f("target_url",               StringType(),    True,  "Tested input with userinfo removed and credential/tracking query values redacted"),
    _f("target_domain",            StringType(),    True,  "Lower-case host without scheme/port/path/leading www"),
    _f("probe_asn",                LongType(),      False, "Numeric probe ASN; 0 when unknown (AS0 / unparseable)"),
    _f("probe_cc",                 StringType(),    False, "ISO-3166 alpha-2 probe country"),
    _f("probe_network_name",       StringType(),    True,  "Probe ISP / network brand name"),
    _f("masked_probe_subnet",      StringType(),    True,  "/24 (IPv4) or /48 (IPv6) mask; NULL when the source IP is loopback/unspecified"),
    _f("hashed_resolver_ip",       StringType(),    True,  "HMAC-SHA256(pepper, resolver_ip); NULL for empty/loopback resolvers"),
    _f("resolver_asn",             LongType(),      False, "Numeric resolver ASN; 0 when unknown"),
    _f("test_name",                StringType(),    False, "OONI nettest name"),
    _f("tampering_vector",         StringType(),    False, "DNS_TAMPERING|TCP_RESET|TLS_DROP|HTTP_BLOCK|ANOMALY_UNCLASSIFIED|BENIGN|UNCLASSIFIED"),
    _f("classification_status",    StringType(),    False, "CONFIRMED_BLOCKED|HEURISTIC_ANOMALY|BENIGN|UNCLASSIFIED (epistemic confidence)"),
    _f("ooni_anomaly",             BooleanType(),   False, "OONI API anomaly flag (NULL upstream treated as false)"),
    _f("ooni_confirmed",           BooleanType(),   False, "OONI API confirmed-blocking flag (NULL upstream treated as false)"),
    _f("is_anomaly",               BooleanType(),   False, "Any interference evidence (OONI or protocol); NOT proof of censorship"),
    _f("dns_anomaly_flag",         IntegerType(),   False, "1 if DNS-layer interference evidence"),
    _f("tcp_anomaly_flag",         IntegerType(),   False, "1 if TCP-layer interference evidence"),
    _f("tls_anomaly_flag",         IntegerType(),   False, "1 if TLS-handshake interference evidence"),
    _f("http_anomaly_flag",        IntegerType(),   False, "1 if HTTP-layer interference evidence"),
    _f("dns_experiment_failure",   StringType(),    True,  "web_connectivity dns_experiment_failure"),
    _f("http_experiment_failure",  StringType(),    True,  "web_connectivity http_experiment_failure"),
    _f("test_runtime_seconds",     DoubleType(),    True,  "Total test harness runtime in seconds (NOT network latency)"),
    _f("content_category",         StringType(),    False, "Citizen Lab category code; MISC when unmatched"),
    _f("category_description",     StringType(),    True,  "Citizen Lab category description"),
    _f("category_match_type",      StringType(),    False, "EXACT_URL|DOMAIN_FALLBACK|UNMATCHED"),
    _f("taxonomy_version",         StringType(),    False, "Pinned Citizen Lab snapshot used for enrichment"),
    _f("human_rights_risk_tier",   StringType(),    False, "HIGH|MEDIUM|LOW derived from content_category"),
    _f("load_timestamp",           TimestampType(), False, "Audit: when the row was inserted / last re-enriched"),
    _f("_batch_id",                StringType(),    False, "Audit: batch that last wrote the row"),
])


In [0]:

# --- 0.2 Child tables ------------------------------------------------------------
SILVER_DNS_SCHEMA = StructType([
    _f("query_id",             StringType(),    False, "PK: SHA-256(measurement_id || query_pos)"),
    _f("measurement_id",       StringType(),    False, "FK to silver_network_measurements"),
    _f("measurement_date",     DateType(),      False, "Partition column"),
    _f("probe_cc",             StringType(),    False, "Probe country"),
    _f("query_pos",            IntegerType(),   False, "Zero-based position inside test_keys.queries"),
    _f("query_type",           StringType(),    True,  "A / AAAA / HTTPS ..."),
    _f("query_hostname",       StringType(),    True,  "Hostname queried"),
    _f("query_engine",         StringType(),    True,  "Resolver engine (system/udp/doh...)"),
    _f("query_failure",        StringType(),    True,  "OONI failure string, NULL on success"),
    _f("is_dns_failure",       BooleanType(),   False, "True when query_failure is present"),
    _f("total_answers_count",  IntegerType(),   False, "Number of answers"),
    _f("ipv4_answers_count",   IntegerType(),   False, "Answers carrying an IPv4 address"),
    _f("cname_answers_count",  IntegerType(),   False, "CNAME answers (answer_type CNAME or hostname set)"),
    _f("load_timestamp",       TimestampType(), False, "Audit load time"),
])
 
SILVER_TCP_SCHEMA = StructType([
    _f("tcp_connect_id",    StringType(),    False, "PK: SHA-256(measurement_id || tcp_pos)"),
    _f("measurement_id",    StringType(),    False, "FK to silver_network_measurements"),
    _f("measurement_date",  DateType(),      False, "Partition column"),
    _f("probe_cc",          StringType(),    False, "Probe country"),
    _f("tcp_pos",           IntegerType(),   False, "Zero-based position inside test_keys.tcp_connect"),
    _f("destination_ip",    StringType(),    True,  "Target server IP (not a probe/activist address)"),
    _f("destination_port",  IntegerType(),   True,  "Target port"),
    _f("is_blocked",        BooleanType(),   False, "OONI status.blocked"),
    _f("failure_reason",    StringType(),    True,  "OONI status.failure"),
    _f("is_success",        BooleanType(),   False, "OONI status.success"),
    _f("load_timestamp",    TimestampType(), False, "Audit load time"),
])
 
SILVER_APP_SCHEMA = StructType([
    _f("app_measurement_id",         StringType(),    False, "PK = measurement_id of the parent row"),
    _f("measurement_date",           DateType(),      False, "Partition column"),
    _f("probe_cc",                   StringType(),    False, "Probe country"),
    _f("probe_asn",                  LongType(),      False, "Probe ASN"),
    _f("app_name",                   StringType(),    False, "whatsapp|telegram|signal|tor|psiphon|facebook_messenger"),
    _f("is_anomaly",                 BooleanType(),   False, "Same value as the parent row"),
    _f("whatsapp_web_status",        StringType(),    True,  "whatsapp_web_status"),
    _f("whatsapp_endpoints_status",  StringType(),    True,  "whatsapp_endpoints_status"),
    _f("telegram_http_blocked",      BooleanType(),   True,  "telegram_http_blocking"),
    _f("telegram_tcp_blocked",       BooleanType(),   True,  "telegram_tcp_blocking"),
    _f("telegram_web_status",        StringType(),    True,  "telegram_web_status"),
    _f("signal_backend_status",      StringType(),    True,  "signal_backend_status"),
    _f("facebook_tcp_blocking",      BooleanType(),   True,  "facebook_tcp_blocking"),
    _f("facebook_dns_blocking",      BooleanType(),   True,  "facebook_dns_blocking"),
    _f("tor_dir_port_accessible",    IntegerType(),   True,  "Tor directory ports reachable"),
    _f("tor_dir_port_total",         IntegerType(),   True,  "Tor directory ports tested"),
    _f("tor_obfs4_accessible",       IntegerType(),   True,  "obfs4 bridges reachable"),
    _f("tor_obfs4_total",            IntegerType(),   True,  "obfs4 bridges tested"),
    _f("psiphon_bootstrap_time",     DoubleType(),    True,  "Psiphon bootstrap time in seconds"),
    _f("app_failure",                StringType(),    True,  "Generic test_keys.failure"),
    _f("load_timestamp",             TimestampType(), False, "Audit load time"),
])
 

In [0]:

# --- 0.3 Bronze input contract (the only columns Silver reads) -------------------
BRONZE_INPUT_CONTRACT: Dict[str, str] = {
    "measurement_uid": "string", "report_id": "string", "input": "string", "test_name": "string",
    "measurement_start_time": "string", "probe_asn": "string", "probe_cc": "string",
    "probe_ip": "string", "probe_network_name": "string", "resolver_asn": "string",
    "resolver_ip": "string", "test_runtime": "double", "test_keys": "string",
    "api_anomaly": "boolean", "api_confirmed": "boolean",
    "measurement_date": "date", "load_timestamp": "timestamp",
}
# Known-but-unused Bronze columns (privacy-dropped or irrelevant); anything else is reported as drift.
BRONZE_KNOWN_UNUSED = {
    "_source_file", "_batch_id", "_corrupt_record", "test_start_time", "probe_id",
    "resolver_network_name", "software_name", "software_version", "test_version",
    "data_format_version", "annotations", "test_helpers", "extensions",
}
# Privacy boundary (Phase 1 step 2.4): never present in any Silver schema.
FORBIDDEN_SILVER_COLUMNS = {
    "probe_id", "probe_ip", "test_keys", "annotations", "test_helpers", "extensions",
    "resolver_ip", "resolver_address",
}

In [0]:

# --- 0.4 test_keys parsing schemas (explicit DDL; all leaves STRING = drift tolerant) ---
# Separate from_json calls per array so one structural surprise cannot null the others.
TEST_KEYS_SCALAR_DDL = (
    "blocking STRING, accessible STRING, dns_experiment_failure STRING, http_experiment_failure STRING, "
    "control_failure STRING, dns_consistency STRING, failure STRING, bootstrap_time STRING, "
    "telegram_http_blocking STRING, telegram_tcp_blocking STRING, telegram_web_status STRING, "
    "whatsapp_endpoints_status STRING, whatsapp_endpoints_dns_inconsistent STRING, whatsapp_web_status STRING, "
    "registration_server_status STRING, signal_backend_status STRING, "
    "facebook_tcp_blocking STRING, facebook_dns_blocking STRING, "
    "dir_port_accessible STRING, dir_port_total STRING, obfs4_accessible STRING, obfs4_total STRING"
)
TEST_KEYS_QUERIES_DDL = (
    "queries ARRAY<STRUCT<failure: STRING, query_type: STRING, hostname: STRING, engine: STRING, "
    "answers: ARRAY<STRUCT<answer_type: STRING, ipv4: STRING, ipv6: STRING, hostname: STRING>>>>"
)
TEST_KEYS_TCP_DDL = (
    "tcp_connect ARRAY<STRUCT<ip: STRING, port: STRING, "
    "status: STRUCT<blocked: STRING, failure: STRING, success: STRING>>>"
)
TEST_KEYS_TLS_DDL = "tls_handshakes ARRAY<STRUCT<failure: STRING>>"
# NOTE: response.body is deliberately NOT parsed: it can be {"data": ..., "format": "base64"} (SILVER-11).
TEST_KEYS_REQUESTS_DDL = "requests ARRAY<STRUCT<failure: STRING, response: STRUCT<code: STRING>>>"
 

In [0]:
 
# --- 0.5 Domain vocabularies ----------------------------------------------------
TAMPERING_VECTORS = ["DNS_TAMPERING", "TCP_RESET", "TLS_DROP", "HTTP_BLOCK",
                     "ANOMALY_UNCLASSIFIED", "BENIGN", "UNCLASSIFIED"]
CLASSIFICATION_STATUSES = ["CONFIRMED_BLOCKED", "HEURISTIC_ANOMALY", "BENIGN", "UNCLASSIFIED"]
CATEGORY_MATCH_TYPES = ["EXACT_URL", "DOMAIN_FALLBACK", "UNMATCHED"]
APP_TEST_NAMES = ["whatsapp", "telegram", "signal", "tor", "psiphon", "facebook_messenger"]
 
RISK_TIER_HIGH = ["POLR", "HUMR", "NEWS", "LGBT", "MMED"]
RISK_TIER_MEDIUM = ["ANON", "COMM", "REL", "FILE", "PUBH"]
# When several categories exist for one domain, political / rights categories win (Phase 2 step 2.2).
CATEGORY_PRIORITY_1 = ["POLR", "HUMR", "NEWS", "LGBT", "REL"]
CATEGORY_PRIORITY_2 = ["COMM", "FILE", "ANON"]
 
QUARANTINE_SOURCE_TAG = "BRONZE_TO_SILVER"
PARENT_MERGE_COMPARE_COLUMNS = [
    "classification_status", "tampering_vector", "is_anomaly", "ooni_anomaly", "ooni_confirmed",
    "dns_anomaly_flag", "tcp_anomaly_flag", "tls_anomaly_flag", "http_anomaly_flag",
    "content_category", "category_description", "category_match_type", "taxonomy_version",
    "human_rights_risk_tier",
]
 

In [0]:
 
# ==============================================================================
# STEP 1: PARAMETERS, SECRETS AND SESSION
# ==============================================================================
 
def init_silver_widgets() -> None:
    """Creates Databricks widgets. Safe no-op outside an interactive notebook."""
    try:
        import IPython
        ipy = IPython.get_ipython()
        if ipy and "dbutils" in ipy.user_ns:
            w = ipy.user_ns["dbutils"].widgets
            yesterday = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
            w.text("batch_date", yesterday, "1. Batch Date T (YYYY-MM-DD)")
            w.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "2. Load Type")
            w.text("lookback_days", str(getattr(config, "DEFAULT_LOOKBACK_DAYS", 3)), "3. Rolling Lookback Days")
            w.text("backfill_start", "", "4. Backfill Start (YYYY-MM-DD, optional)")
            w.text("backfill_end", "", "5. Backfill End (YYYY-MM-DD, optional)")
            w.text("country_filter", "ALL", "6. Country Filter (ALL or PK/IR/RU)")
    except Exception:
        pass


In [0]:

def _parse_date(value: Any, name: str) -> date:
    try:
        return datetime.strptime(str(value).strip(), "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"[Parameter Error] '{name}' must be YYYY-MM-DD, got '{value}'.")
 
 
def resolve_silver_parameters(
    batch_date: Optional[str] = None, load_type: Optional[str] = None,
    lookback_days: Optional[int] = None, backfill_start: Optional[str] = None,
    backfill_end: Optional[str] = None, country_filter: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Resolves parameters (explicit args > widgets > env > defaults) and the Bronze window to process.
      incremental : [batch_date - lookback_days, batch_date]
      backfill    : [backfill_start, backfill_end] if given, else the lookback window
                    (lookback_days=0 re-processes exactly one historical date)
      full        : the entire Bronze table
    """
    get = config.get_pipeline_parameter
    yesterday = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
 
    r_load = str(load_type or get("load_type", "incremental")).lower()
    if r_load not in ("full", "incremental", "backfill"):
        raise ValueError(f"[Parameter Error] load_type must be full|incremental|backfill, got '{r_load}'.")
    r_batch = _parse_date(batch_date or get("batch_date", yesterday), "batch_date")
    r_lookback = int(lookback_days if lookback_days is not None
                     else get("lookback_days", getattr(config, "DEFAULT_LOOKBACK_DAYS", 3)))
    if r_lookback < 0:
        raise ValueError("[Parameter Error] lookback_days must be >= 0.")
    r_country = str(country_filter or get("country_filter", "ALL")).strip().upper() or "ALL"
    r_bf_start = str(backfill_start or get("backfill_start", "") or "").strip()
    r_bf_end = str(backfill_end or get("backfill_end", "") or "").strip()
 
    if r_load == "full":
        win_start = win_end = None
        batch_id = f"SILVER_FULL_{r_country}"
    else:
        if r_load == "backfill" and r_bf_start:
            win_start = _parse_date(r_bf_start, "backfill_start")
            win_end = _parse_date(r_bf_end, "backfill_end") if r_bf_end else win_start
        else:
            win_end = r_batch
            win_start = r_batch - timedelta(days=r_lookback)
        if win_start > win_end:
            raise ValueError(f"[Parameter Error] window start {win_start} is after end {win_end}.")
        batch_id = f"SILVER_{win_start:%Y%m%d}_{win_end:%Y%m%d}_{r_country}_{r_load.upper()}"
 
    return {
        "batch_date": r_batch.isoformat(), "load_type": r_load, "lookback_days": r_lookback,
        "window_start": win_start.isoformat() if win_start else None,
        "window_end": win_end.isoformat() if win_end else None,
        "country_filter": r_country, "batch_id": batch_id,
    }

In [0]:
import os
os.environ["PEPPER_HASH_SALT"] = "fe5931fc7a166f716f5e523b79bd01ac9114b6baa4699938e6b77bd53f1ce676"

In [0]:
 
def resolve_pepper_salt(explicit: Optional[str] = None, allow_dev_fallback: bool = False) -> str:
    """
    FAIL-CLOSED pepper resolution (Phase 1 step 2.2, checklist item 12).
    Order: explicit argument > env PEPPER_HASH_SALT > env PANOPTICON_SALT > Databricks secret
    (scope 'panopticon_vault', key 'ip_salt'). There is NO hard-coded production fallback:
    if nothing is found the pipeline stops. `allow_dev_fallback=True` exists for local tests only.
    """
    candidate = explicit or os.environ.get("PEPPER_HASH_SALT") or os.environ.get("PANOPTICON_SALT")
    if not candidate:
        try:
            import IPython
            dbutils = IPython.get_ipython().user_ns.get("dbutils")
            if dbutils:
                candidate = dbutils.secrets.get(scope="panopticon_vault", key="ip_salt")
        except Exception:
            candidate = None
    if not candidate:
        if allow_dev_fallback:
            return "DEV_ONLY_NOT_FOR_PRODUCTION"
        raise ValueError("[Security Guard] No pepper salt available (PEPPER_HASH_SALT / secret "
                         "panopticon_vault/ip_salt). Refusing to run with a hard-coded fallback.")
    if not str(candidate).strip():
        raise ValueError("[Security Guard] PEPPER_HASH_SALT cannot be empty.")
    return str(candidate)
 
 
def _safe_conf_set(conf: Any, key: str, value: str) -> None:
    try:
        conf.set(key, value)
    except Exception:
        pass   # Serverless rejects some session configs; they are optional tuning
 
 
def resolve_spark_session(spark: Optional[Any] = None) -> Any:
    """Active session + Community-Edition safeguards (Phase 5 step 2.1) + UTC timestamp semantics."""
    session = spark or SparkSession.builder.getOrCreate()
    try:
        config.configure_spark_runtime(session)
    except Exception:
        pass
    _safe_conf_set(session.conf, "spark.sql.shuffle.partitions", "4")
    _safe_conf_set(session.conf, "spark.sql.session.timeZone", "UTC")
    _safe_conf_set(session.conf, "spark.sql.parquet.compression.codec", "snappy")
    _safe_conf_set(session.conf, "spark.databricks.delta.optimizeWrite.enabled", "true")
    _safe_conf_set(session.conf, "spark.databricks.delta.autoCompact.enabled", "true")
    _safe_conf_set(session.conf, "spark.databricks.delta.schema.autoMerge.enabled", "true")
    return session
 

In [0]:

def _table_path(key: str, default_name: str) -> str:
    """Path for a Silver table: config.PATHS[key] if defined, else a sibling of the parent table."""
    paths = getattr(config, "PATHS", {}) or {}
    if paths.get(key):
        return paths[key]
    return paths["silver"].rstrip("/").rsplit("/", 1)[0] + "/" + default_name
 
 
def _table_name(key: str, default_name: str) -> str:
    return (getattr(config, "TABLE_NAMES", {}) or {}).get(key, default_name)

In [0]:

# ==============================================================================
# STEP 2: READ BRONZE + SCHEMA-DRIFT GUARD
# ==============================================================================
 
class SchemaDriftError(RuntimeError):
    """Raised when Bronze lacks a column Silver cannot work without."""
 
 
def detect_and_absorb_schema_drift(df_bronze: "DataFrame") -> Tuple["DataFrame", Dict[str, Any]]:
    """
    Compares live Bronze with BRONZE_INPUT_CONTRACT.
      missing column -> SchemaDriftError (also the guard that Bronze Phase 0 was re-run)
      type changed   -> try_cast coercion; bad values become NULL and are handled by the DQ rules
      extra column   -> ignored (explicit projection) and reported
    """
    actual = {f.name: f.dataType.simpleString() for f in df_bronze.schema.fields}
    missing = sorted(set(BRONZE_INPUT_CONTRACT) - set(actual))
    if missing:
        hint = (" Re-run Bronze ingestion (Phase 0) before Silver." if
                {"api_anomaly", "api_confirmed", "measurement_uid"} & set(missing) else "")
        raise SchemaDriftError(f"Bronze is missing required columns: {missing}.{hint}")
 
    changed = {c: (actual[c], t) for c, t in BRONZE_INPUT_CONTRACT.items() if actual[c] != t}
    unexpected = sorted(set(actual) - set(BRONZE_INPUT_CONTRACT) - BRONZE_KNOWN_UNUSED)
    projected = df_bronze.select(*[
        (F.col(c) if actual[c] == t else F.expr(f"try_cast(`{c}` AS {t})")).alias(c)
        for c, t in BRONZE_INPUT_CONTRACT.items()
    ])
    if changed or unexpected:
        print(f"[Schema Drift] type_changes={changed} unexpected_columns={unexpected} (absorbed, batch continues)")
    return projected, {"type_changes": changed, "unexpected_columns": unexpected}
 
 
def read_bronze_window(spark, bronze_path: str, window_start: Optional[str],
                       window_end: Optional[str], country_filter: str) -> "DataFrame":
    """Partition-pruned read (Bronze is partitioned by probe_cc, measurement_date)."""
    df = spark.read.format("delta").load(bronze_path)
    if window_start and window_end:
        df = df.filter((F.col("measurement_date") >= F.lit(window_start).cast("date")) &
                       (F.col("measurement_date") <= F.lit(window_end).cast("date")))
    if country_filter != "ALL":
        df = df.filter(F.upper(F.col("probe_cc")) == country_filter)
    return df

In [0]:

def read_taxonomy(spark) -> "DataFrame":
    """Citizen Lab reference. Degrades to empty (everything UNMATCHED/MISC) with a warning; a later run
    re-enriches existing rows through the MERGE once the taxonomy exists."""
    path = config.PATHS["reference"]
    try:
        if DeltaTable.isDeltaTable(spark, path):
            return spark.read.format("delta").load(path)
    except Exception:
        pass
    print(f"[Taxonomy Warning] No reference table at {path}; all rows will be UNMATCHED until it exists.")
    return spark.createDataFrame([], StructType([
        StructField("url", StringType()), StructField("category_code", StringType()),
        StructField("category_description", StringType()), StructField("country_code", StringType()),
        StructField("taxonomy_version", StringType())]))
 
 
def current_taxonomy_version(df_ref: "DataFrame") -> str:
    """Single reproducibility label of the snapshot used (also stamped on UNMATCHED rows)."""
    if "taxonomy_version" not in df_ref.columns:
        return "UNKNOWN"
    row = df_ref.agg(F.max("taxonomy_version").alias("v")).first()
    return (row["v"] if row and row["v"] else "UNKNOWN")
 

In [0]:

# ==============================================================================
# STEP 3: PURE TRANSFORMATION FUNCTIONS (no I/O; unit-testable)
# ==============================================================================
 
# ---- 3.1 ANSI-safe parsing helpers -------------------------------------------------
 
def parse_utc_timestamp(c: "F.Column") -> "F.Column":
    return F.coalesce(
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss.SSS")),
    )
 
 
def parse_asn(c: "F.Column") -> "F.Column":
    """'AS9541' / '9541' -> 9541; unknown / 'AS0' / unparseable -> 0 (never NULL: Phase 1 step 2.3)."""
    digits = F.regexp_extract(F.upper(F.trim(c)), r"^(?:AS)?(\d+)$", 1)
    parsed = F.when((F.length(digits) > 0) & (F.length(digits) <= 18), digits).cast("long")
    return F.coalesce(parsed, F.lit(0).cast("long"))
 
 
def _to_bool(c: "F.Column") -> "F.Column":
    lc = F.lower(F.trim(c))
    return F.when(lc == "true", F.lit(True)).when(lc == "false", F.lit(False))
 
 
def _to_int(c: "F.Column") -> "F.Column":
    t = F.trim(c)
    return F.when(t.rlike(r"^-?\d{1,9}$"), t.cast("int"))
 
 
def _to_double(c: "F.Column") -> "F.Column":
    t = F.trim(c)
    return F.when(t.rlike(r"^-?\d+(\.\d+)?([eE][-+]?\d+)?$"), t.cast("double"))
 
 
def _present(c: "F.Column") -> "F.Column":
    """True when a failure-style string carries a real value (not NULL / 'null' / '')."""
    return c.isNotNull() & (F.trim(c) != "") & (F.lower(F.trim(c)) != "null")
 
 
def _b(c: "F.Column") -> "F.Column":
    return F.coalesce(c, F.lit(False))

In [0]:

# ---- 3.2 Phase 1: identity + privacy ---------------------------------------------------
 
def _key_part(c: "F.Column") -> "F.Column":
    """Length-prefixed component: NULL, empty and delimiter-containing values can never collide."""
    return F.when(c.isNull(), F.lit("~N")).otherwise(F.concat(F.lit("V"), F.length(c).cast("string"), F.lit(":"), c))
 
 
def generate_measurement_id(df: "DataFrame") -> "DataFrame":
    """
    measurement_id = OONI measurement_uid when present, else SHA-256 over the 6 Bronze natural-key parts
    (probe_cc, measurement_date, report_id, measurement_start_time, test_name, input).
    """
    fallback = F.sha2(F.concat_ws("|", *[_key_part(F.col(c).cast("string")) for c in
                      ("probe_cc", "measurement_date", "report_id", "measurement_start_time", "test_name", "input")]), 256)
    uid = F.trim(F.col("measurement_uid"))
    return df.withColumn("measurement_id", F.when(uid.isNotNull() & (F.length(uid) > 0), uid).otherwise(fallback))
 
 
_SENSITIVE_QUERY_PARAMS = (
    r"(?:token|access_token|refresh_token|id_token|auth|authorization|apikey|api_key|key|"
    r"password|passwd|pwd|secret|session|session_id|sessionid|sid|jwt|signature|sig|code|"
    r"fbclid|gclid|utm_[a-z0-9_]+)"
)
 
 
def sanitize_url(c: "F.Column") -> "F.Column":
    no_userinfo = F.regexp_replace(c, r"^([A-Za-z][A-Za-z0-9+.\-]*://)[^/?#@]*@", "$1")
    return F.regexp_replace(no_userinfo, r"(?i)([?&]" + _SENSITIVE_QUERY_PARAMS + r"=)[^&#]*", "$1REDACTED")
 
 
def mask_probe_subnet(ip_col: "F.Column") -> "F.Column":
    """IPv4 -> a.b.c.0/24, IPv6 (>=3 explicit hextets) -> h1:h2:h3::/48. Loopback/unspecified -> NULL
    (OONI scrubs probe_ip to 127.0.0.1, so a constant '127.0.0.0/24' would be meaningless)."""
    ip = F.trim(ip_col)
    uninformative = ip.rlike(r"^(127\.\d{1,3}\.\d{1,3}\.\d{1,3}|0\.0\.0\.0|::1?)$")
    v4 = F.when(ip.rlike(r"^\d{1,3}(\.\d{1,3}){3}$"),
                F.regexp_replace(ip, r"^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.\d{1,3}$", "$1.$2.$3.0/24"))
    v6 = F.when(ip.rlike(r"^[0-9a-fA-F]{1,4}(:[0-9a-fA-F]{1,4}){2,}"),
                F.regexp_replace(ip, r"^([0-9a-fA-F]{1,4}):([0-9a-fA-F]{1,4}):([0-9a-fA-F]{1,4}).*$", "$1:$2:$3::/48"))
    return F.when(uninformative, F.lit(None).cast("string")).otherwise(F.coalesce(v4, v6))
 
 
def make_hmac_udf(pepper: str):
    """HMAC-SHA256(key = pepper, message = ip): matches the course spec 'Salted HMAC-SHA256'."""
    if not pepper:
        raise ValueError("[Security Guard] PEPPER_HASH_SALT cannot be empty in production.")
    key = pepper.encode("utf-8")
 
    def _hmac(ip: Optional[str]) -> Optional[str]:
        if ip is None or not ip.strip():
            return None
        return hmac.new(key, ip.strip().encode("utf-8"), hashlib.sha256).hexdigest()
 
    return F.udf(_hmac, StringType())
 
 
def hash_resolver_ip(df: "DataFrame", pepper: str) -> "DataFrame":
    hmac_udf = make_hmac_udf(pepper)
    usable = (F.col("resolver_ip").isNotNull() & (F.trim(F.col("resolver_ip")) != "")
              & (F.trim(F.col("resolver_ip")) != "127.0.0.1"))
    return df.withColumn("hashed_resolver_ip",
                         F.when(usable, hmac_udf(F.col("resolver_ip"))).otherwise(F.lit(None).cast("string")))
 

In [0]:

# ---- 3.3 Phase 2: URL normalisation + two-tier Citizen Lab enrichment --------------------------
 
def normalize_url_key(url_col: "F.Column") -> "F.Column":
    """Comparable URL identity: lower-case, no scheme / userinfo / query / fragment / leading www / trailing slash.
    NULL for missing, empty or 'NO_INPUT' inputs (non-web tests)."""
    u = F.lower(F.trim(url_col))
    k = F.regexp_replace(u, r"^[a-z][a-z0-9+.\-]*://", "")
    k = F.regexp_replace(k, r"^[^/?#@]*@", "")
    k = F.regexp_replace(k, r"[?#].*$", "")
    k = F.regexp_replace(k, r"^www\.", "")
    k = F.regexp_replace(k, r"/+$", "")
    valid = url_col.isNotNull() & (F.trim(url_col) != "") & (F.trim(url_col) != "NO_INPUT") & (F.length(k) > 0)
    return F.when(valid, k)
 
 
def host_from_key(key_col: "F.Column") -> "F.Column":
    host = F.regexp_extract(key_col, r"^([^/:]+)", 1)
    return F.when(key_col.isNotNull() & (F.length(host) > 0) & ~key_col.startswith("["), host)
 
 
def _category_priority(c: "F.Column") -> "F.Column":
    return (F.when(c.isin(*CATEGORY_PRIORITY_1), F.lit(1))
             .when(c.isin(*CATEGORY_PRIORITY_2), F.lit(2)).otherwise(F.lit(3)))
 
 
def prepare_taxonomy(df_ref: "DataFrame") -> "DataFrame":
    cols = df_ref.columns
    country = F.upper(F.trim(F.col("country_code"))) if "country_code" in cols else F.lit("GLOBAL")
    version = F.col("taxonomy_version") if "taxonomy_version" in cols else F.lit("UNKNOWN")
    base = df_ref.select(
        normalize_url_key(F.col("url")).alias("ref_key"),
        F.upper(F.trim(F.col("category_code"))).alias("ref_category"),
        F.col("category_description").alias("ref_description"),
        country.alias("ref_country"),
        version.alias("ref_version"))
    base = base.filter(F.col("ref_key").isNotNull() & F.col("ref_category").isNotNull())
    return (base.withColumn("ref_host", host_from_key(F.col("ref_key")))
                .withColumn("ref_priority", _category_priority(F.col("ref_category"))))
 
 
def enrich_citizenlab_taxonomy(df: "DataFrame", df_ref_prepared: "DataFrame", default_version: str) -> "DataFrame":
    """
    Tier 1 EXACT_URL       : normalised URL key equality
    Tier 2 DOMAIN_FALLBACK : longest matching parent-domain suffix (sub.twitter.com -> twitter.com),
                             one reference row per (domain, country) chosen by category priority
    Tier 3 UNMATCHED       : category MISC (rows are never dropped; null inputs fall here)
    A reference entry for the probe's own country beats a GLOBAL entry. Expects columns:
    measurement_id, probe_cc, _url_key, target_domain.
    """
    from pyspark.sql.functions import broadcast
    slim = df.select("measurement_id", "probe_cc", "_url_key", "target_domain")
    country_match = (F.col("ref_country") == F.col("probe_cc")).desc()
 
    # ---- Tier 1
    t1 = (slim.where(F.col("_url_key").isNotNull())
          .join(broadcast(df_ref_prepared), F.col("_url_key") == F.col("ref_key"), "inner"))
    w1 = Window.partitionBy("measurement_id").orderBy(country_match, F.col("ref_priority").asc(), F.col("ref_category").asc())
    t1 = (t1.withColumn("_rk", F.row_number().over(w1)).where(F.col("_rk") == 1)
            .select("measurement_id", F.col("ref_category").alias("cat1"),
                    F.col("ref_description").alias("desc1"), F.col("ref_version").alias("ver1")))
 
    # ---- Tier 2 (reference side deduplicated per domain+country, so the join cannot fan out)
    w_dom = Window.partitionBy("ref_host", "ref_country").orderBy(F.col("ref_priority").asc(), F.col("ref_category").asc())
    ref_dom = (df_ref_prepared.where(F.col("ref_host").isNotNull())
               .withColumn("_rd", F.row_number().over(w_dom)).where(F.col("_rd") == 1).drop("_rd"))
    parts = slim.where(F.col("target_domain").isNotNull()).withColumn("_parts", F.split(F.col("target_domain"), r"\."))
    cands = parts.select(
        "measurement_id", "probe_cc",
        F.posexplode(F.expr("transform(sequence(0, greatest(size(_parts) - 2, 0)), "
                            "i -> array_join(slice(_parts, i + 1, size(_parts) - i), '.'))")).alias("cand_pos", "cand_domain"))
    t2 = cands.join(broadcast(ref_dom), F.col("cand_domain") == F.col("ref_host"), "inner")
    w2 = Window.partitionBy("measurement_id").orderBy(
        F.col("cand_pos").asc(), country_match, F.col("ref_priority").asc(), F.col("ref_category").asc())
    t2 = (t2.withColumn("_rk", F.row_number().over(w2)).where(F.col("_rk") == 1)
            .select("measurement_id", F.col("ref_category").alias("cat2"),
                    F.col("ref_description").alias("desc2"), F.col("ref_version").alias("ver2")))
 
    joined = df.join(t1, "measurement_id", "left").join(t2, "measurement_id", "left")
    return (joined
        .withColumn("category_match_type",
                    F.when(F.col("cat1").isNotNull(), F.lit("EXACT_URL"))
                     .when(F.col("cat2").isNotNull(), F.lit("DOMAIN_FALLBACK")).otherwise(F.lit("UNMATCHED")))
        .withColumn("content_category", F.coalesce(F.col("cat1"), F.col("cat2"), F.lit("MISC")))
        .withColumn("category_description", F.coalesce(F.col("desc1"), F.col("desc2"), F.lit("Uncategorized Content")))
        .withColumn("taxonomy_version", F.coalesce(F.col("ver1"), F.col("ver2"), F.lit(default_version)))
        .drop("cat1", "desc1", "ver1", "cat2", "desc2", "ver2"))
 
 
def assign_human_rights_risk_tier(df: "DataFrame") -> "DataFrame":
    return df.withColumn("human_rights_risk_tier",
        F.when(F.col("content_category").isin(*RISK_TIER_HIGH), F.lit("HIGH"))
         .when(F.col("content_category").isin(*RISK_TIER_MEDIUM), F.lit("MEDIUM"))
         .otherwise(F.lit("LOW")))

In [0]:

# ---- 3.4 Phase 3: protocol signals + epistemic classification --------------------------------
 
def parse_test_keys(df: "DataFrame") -> "DataFrame":
    """Adds sc/qs/tcs/tls/rqs structs. Independent parses: a structural surprise in one array cannot null the rest."""
    opts = {"mode": "PERMISSIVE"}
    return (df.withColumn("sc", F.from_json(F.col("test_keys"), TEST_KEYS_SCALAR_DDL, opts))
              .withColumn("qs", F.from_json(F.col("test_keys"), TEST_KEYS_QUERIES_DDL, opts))
              .withColumn("tcs", F.from_json(F.col("test_keys"), TEST_KEYS_TCP_DDL, opts))
              .withColumn("tls", F.from_json(F.col("test_keys"), TEST_KEYS_TLS_DDL, opts))
              .withColumn("rqs", F.from_json(F.col("test_keys"), TEST_KEYS_REQUESTS_DDL, opts)))
 
 
def add_signals_and_classification(df: "DataFrame") -> "DataFrame":
    """
    Evidence (4 protocol flags) is decoupled from confidence (classification_status).
    Requires: test_name, api_anomaly, api_confirmed, test_keys. Adds flags, ooni_*, is_anomaly,
    classification_status, tampering_vector (+ parsed structs sc/qs/tcs/tls/rqs).
    """
    df = parse_test_keys(df)
    tn = F.col("test_name")
    low = lambda name: F.lower(F.col(f"sc.{name}"))   # noqa: E731
    blocking = low("blocking")
 
    # ---- generic OONI signals (web_connectivity-style arrays + summary field)
    dns_generic = (_b(_present(F.col("sc.dns_experiment_failure")))
                   | _b(F.exists(F.col("qs.queries"), lambda q: _present(q["failure"])))
                   | _b(blocking == "dns"))
    tcp_generic = (_b(F.exists(F.col("tcs.tcp_connect"),
                               lambda t: (F.lower(t["status"]["blocked"]) == "true") | _present(t["status"]["failure"])))
                   | _b(blocking.isin("tcp_ip", "tcp-ip")))
    tls_generic = _b(F.exists(F.col("tls.tls_handshakes"), lambda h: _present(h["failure"])))
    http_generic = (_b(_present(F.col("sc.http_experiment_failure")))
                    | _b(F.exists(F.col("rqs.requests"), lambda r: r["response"]["code"].isin("403", "451")))
                    | _b(blocking.isin("http-failure", "http-diff")))
 
    # ---- application nettests (no web arrays; dedicated summary keys)
    app_dns = ((tn == "whatsapp") & _b(~low("whatsapp_endpoints_dns_inconsistent").isin("", "[]"))) | \
              ((tn == "facebook_messenger") & _b(low("facebook_dns_blocking") == "true"))
    app_tcp = ((tn == "telegram") & _b(low("telegram_tcp_blocking") == "true")) | \
              ((tn == "whatsapp") & _b(low("whatsapp_endpoints_status") == "blocked")) | \
              ((tn == "facebook_messenger") & _b(low("facebook_tcp_blocking") == "true"))
    app_tls = (tn == "signal") & _b(low("signal_backend_status") == "blocked")
    app_http = ((tn == "telegram") & (_b(low("telegram_http_blocking") == "true") | _b(low("telegram_web_status") == "blocked"))) | \
               ((tn == "whatsapp") & (_b(low("registration_server_status") == "failed") | _b(low("whatsapp_web_status") == "blocked")))
 
    dns, tcp = _b(dns_generic | app_dns), _b(tcp_generic | app_tcp)
    tls, http = _b(tls_generic | app_tls), _b(http_generic | app_http)
 
    ooni_anomaly = F.coalesce(F.col("api_anomaly"), F.lit(False))        # SILVER-45: explicit NULL handling
    ooni_confirmed = F.coalesce(F.col("api_confirmed"), F.lit(False))
    heuristic = dns | tcp | tls | http
    is_anomaly = ooni_confirmed | ooni_anomaly | heuristic
    # from_json returns a struct of NULLs (not a NULL struct) for malformed JSON, so parseability is tested directly.
    payload_parseable = F.col("test_keys").isNotNull() & F.get_json_object(F.col("test_keys"), "$").isNotNull()
    has_basis = F.col("api_anomaly").isNotNull() | payload_parseable   # ground truth or a parseable payload
 
    status = (F.when(ooni_confirmed, F.lit("CONFIRMED_BLOCKED"))
               .when(is_anomaly, F.lit("HEURISTIC_ANOMALY"))
               .when(has_basis, F.lit("BENIGN"))
               .otherwise(F.lit("UNCLASSIFIED")))
 
    confirmed_vector = (F.when(blocking == "dns", F.lit("DNS_TAMPERING"))
                         .when(blocking.isin("tcp_ip", "tcp-ip"), F.lit("TCP_RESET"))
                         .otherwise(F.lit("HTTP_BLOCK")))            # blockpage fingerprint default
    vector = (F.when(ooni_confirmed, confirmed_vector)
               .when(dns, F.lit("DNS_TAMPERING"))                    # DNS precedes transport
               .when(tcp, F.lit("TCP_RESET"))                        # transport precedes TLS
               .when(tls, F.lit("TLS_DROP"))                         # TLS precedes HTTP
               .when(http, F.lit("HTTP_BLOCK"))
               .when(is_anomaly, F.lit("ANOMALY_UNCLASSIFIED"))
               .when(has_basis, F.lit("BENIGN"))
               .otherwise(F.lit("UNCLASSIFIED")))
 
    return (df
        .withColumn("dns_anomaly_flag", dns.cast("int")).withColumn("tcp_anomaly_flag", tcp.cast("int"))
        .withColumn("tls_anomaly_flag", tls.cast("int")).withColumn("http_anomaly_flag", http.cast("int"))
        .withColumn("ooni_anomaly", ooni_anomaly).withColumn("ooni_confirmed", ooni_confirmed)
        .withColumn("is_anomaly", is_anomaly)
        .withColumn("classification_status", status).withColumn("tampering_vector", vector))
 

In [0]:

# ---- 3.5 Staging: type-casting, data quality, key, de-duplication ---------------------------
 
def stage_measurements(df_bronze: "DataFrame") -> Tuple["DataFrame", "DataFrame"]:
    """Returns (valid_staged, invalid). Valid rows carry the canonical measurement_id and are unique on it."""
    df = (df_bronze
        .withColumn("_bronze_load_ts", F.col("load_timestamp"))
        .withColumn("event_timestamp", parse_utc_timestamp(F.col("measurement_start_time")))
        .withColumn("silver_date", F.to_date(F.col("event_timestamp")))
        .withColumn("probe_cc_n", F.upper(F.trim(F.col("probe_cc")))))
 
    dq = (F.when(F.col("report_id").isNull() | (F.trim(F.col("report_id")) == ""), "MISSING_REPORT_ID")
           .when(F.col("test_name").isNull() | (F.trim(F.col("test_name")) == ""), "MISSING_TEST_NAME")
           .when(F.col("event_timestamp").isNull(), "UNPARSEABLE_MEASUREMENT_START_TIME")
           .when(~_b(F.col("probe_cc_n").rlike("^[A-Z]{2}$")), "INVALID_PROBE_CC"))
    df = df.withColumn("_dq_reason", dq)
 
    invalid = (df.filter(F.col("_dq_reason").isNotNull())
        .select(F.to_json(F.struct("measurement_uid", "report_id", "input", "test_name", "measurement_start_time",
                                   "probe_asn", "probe_cc", "probe_network_name", "resolver_asn",
                                   "test_runtime")).alias("raw_record"), "_dq_reason"))   # no IPs, no test_keys
 
    valid = (df.filter(F.col("_dq_reason").isNull())
        .withColumn("probe_cc", F.col("probe_cc_n"))
        .withColumn("measurement_date", F.col("silver_date")))
    valid = generate_measurement_id(valid)
    w = Window.partitionBy("measurement_id").orderBy(
        F.when(F.col("test_keys").isNotNull() & (F.col("test_keys") != ""), 1).otherwise(0).desc(),
        F.col("_bronze_load_ts").desc_nulls_last(), F.col("test_runtime").desc_nulls_last())
    valid = valid.withColumn("_dr", F.row_number().over(w)).filter(F.col("_dr") == 1).drop("_dr")
    return valid, invalid
 

In [0]:

# ---- 3.6 Parent + child builders --------------------------------------------------------------
 
def build_parent(staged: "DataFrame", df_taxonomy: "DataFrame", pepper: str, batch_id: str, load_ts: datetime) -> "DataFrame":
    df = (staged
        .withColumn("probe_asn", parse_asn(F.col("probe_asn")))
        .withColumn("resolver_asn", parse_asn(F.col("resolver_asn")))
        .withColumn("target_url", sanitize_url(F.col("input")))
        .withColumn("_url_key", normalize_url_key(F.col("target_url")))
        .withColumn("target_domain", host_from_key(F.col("_url_key")))
        .withColumn("masked_probe_subnet", mask_probe_subnet(F.col("probe_ip"))))
    df = hash_resolver_ip(df, pepper)
    df = add_signals_and_classification(df)
    df = (df
        .withColumn("dns_experiment_failure", F.when(F.col("test_name") == "web_connectivity", F.col("sc.dns_experiment_failure")))
        .withColumn("http_experiment_failure", F.when(F.col("test_name") == "web_connectivity", F.col("sc.http_experiment_failure")))
        .withColumn("test_runtime_seconds", F.col("test_runtime").cast("double")))
    df = enrich_citizenlab_taxonomy(df, prepare_taxonomy(df_taxonomy), current_taxonomy_version(df_taxonomy))
    df = assign_human_rights_risk_tier(df)
    df = (df.withColumn("load_timestamp", F.lit(load_ts).cast("timestamp")).withColumn("_batch_id", F.lit(batch_id)))
    return df.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in SILVER_SCHEMA.fields])
 
 
def build_dns_child(staged: "DataFrame", load_ts: datetime) -> "DataFrame":
    """Independent single-pass projection (never chained with other explodes); resolver addresses are not extracted."""
    df = staged.filter(F.col("test_keys").isNotNull()).withColumn(
        "qs", F.from_json(F.col("test_keys"), TEST_KEYS_QUERIES_DDL, {"mode": "PERMISSIVE"}))
    ex = df.select("measurement_id", "measurement_date", "probe_cc",
                   F.posexplode_outer(F.col("qs.queries")).alias("query_pos", "q")).filter(F.col("q").isNotNull())
    answers = F.col("q.answers")
    n_answers = F.when(answers.isNotNull(), F.size(answers)).otherwise(F.lit(0))
    return ex.select(
        F.sha2(F.concat_ws("||", F.col("measurement_id"), F.col("query_pos").cast("string")), 256).alias("query_id"),
        "measurement_id", "measurement_date", "probe_cc", F.col("query_pos").cast("int").alias("query_pos"),
        F.col("q.query_type").alias("query_type"), F.col("q.hostname").alias("query_hostname"),
        F.col("q.engine").alias("query_engine"), F.col("q.failure").alias("query_failure"),
        _present(F.col("q.failure")).alias("is_dns_failure"),
        n_answers.cast("int").alias("total_answers_count"),
        F.when(answers.isNotNull(), F.size(F.filter(answers, lambda a: a["ipv4"].isNotNull()))).otherwise(F.lit(0)).cast("int").alias("ipv4_answers_count"),
        F.when(answers.isNotNull(), F.size(F.filter(
            answers, lambda a: (F.upper(a["answer_type"]) == "CNAME") | (a["hostname"].isNotNull())))).otherwise(F.lit(0)).cast("int").alias("cname_answers_count"),
        F.lit(load_ts).cast("timestamp").alias("load_timestamp"))
 
 
def build_tcp_child(staged: "DataFrame", load_ts: datetime) -> "DataFrame":
    df = staged.filter(F.col("test_keys").isNotNull()).withColumn(
        "tcs", F.from_json(F.col("test_keys"), TEST_KEYS_TCP_DDL, {"mode": "PERMISSIVE"}))
    ex = df.select("measurement_id", "measurement_date", "probe_cc",
                   F.posexplode_outer(F.col("tcs.tcp_connect")).alias("tcp_pos", "t")).filter(F.col("t").isNotNull())
    return ex.select(
        F.sha2(F.concat_ws("||", F.col("measurement_id"), F.col("tcp_pos").cast("string")), 256).alias("tcp_connect_id"),
        "measurement_id", "measurement_date", "probe_cc", F.col("tcp_pos").cast("int").alias("tcp_pos"),
        F.col("t.ip").alias("destination_ip"), _to_int(F.col("t.port")).alias("destination_port"),
        F.coalesce(_to_bool(F.col("t.status.blocked")), F.lit(False)).alias("is_blocked"),
        F.col("t.status.failure").alias("failure_reason"),
        F.coalesce(_to_bool(F.col("t.status.success")), F.lit(False)).alias("is_success"),
        F.lit(load_ts).cast("timestamp").alias("load_timestamp"))
 
 
def build_app_child(staged: "DataFrame", load_ts: datetime) -> "DataFrame":
    """Polymorphic routing: messaging / circumvention nettests (the parent row is kept as well)."""
    df = staged.filter(F.col("test_name").isin(*APP_TEST_NAMES)).withColumn("probe_asn", parse_asn(F.col("probe_asn")))
    df = add_signals_and_classification(df)
    sc = lambda name: F.col(f"sc.{name}")   # noqa: E731
    return df.select(
        F.col("measurement_id").alias("app_measurement_id"), "measurement_date", "probe_cc", "probe_asn",
        F.col("test_name").alias("app_name"), F.col("is_anomaly"),
        sc("whatsapp_web_status").alias("whatsapp_web_status"),
        sc("whatsapp_endpoints_status").alias("whatsapp_endpoints_status"),
        _to_bool(sc("telegram_http_blocking")).alias("telegram_http_blocked"),
        _to_bool(sc("telegram_tcp_blocking")).alias("telegram_tcp_blocked"),
        sc("telegram_web_status").alias("telegram_web_status"),
        sc("signal_backend_status").alias("signal_backend_status"),
        _to_bool(sc("facebook_tcp_blocking")).alias("facebook_tcp_blocking"),
        _to_bool(sc("facebook_dns_blocking")).alias("facebook_dns_blocking"),
        _to_int(sc("dir_port_accessible")).alias("tor_dir_port_accessible"),
        _to_int(sc("dir_port_total")).alias("tor_dir_port_total"),
        _to_int(sc("obfs4_accessible")).alias("tor_obfs4_accessible"),
        _to_int(sc("obfs4_total")).alias("tor_obfs4_total"),
        F.when(F.col("test_name") == "psiphon", _to_double(sc("bootstrap_time"))).alias("psiphon_bootstrap_time"),
        sc("failure").alias("app_failure"),
        F.lit(load_ts).cast("timestamp").alias("load_timestamp"))
 
 
def transform_bronze_to_silver(
    df_bronze: "DataFrame", df_taxonomy: "DataFrame", pepper: str, batch_id: str, load_ts: datetime,
) -> Dict[str, "DataFrame"]:
    """Returns {'parent','dns','tcp','app','invalid'}; every Silver frame conforms to its *_SCHEMA."""
    staged, invalid = stage_measurements(df_bronze)
    dns = build_dns_child(staged, load_ts)
    tcp = build_tcp_child(staged, load_ts)
    app = build_app_child(staged, load_ts)
    return {
        "parent": build_parent(staged, df_taxonomy, pepper, batch_id, load_ts),
        "dns": dns.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in SILVER_DNS_SCHEMA.fields]),
        "tcp": tcp.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in SILVER_TCP_SCHEMA.fields]),
        "app": app.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in SILVER_APP_SCHEMA.fields]),
        "invalid": invalid,
    }
 

In [0]:

# COMMAND ----------
 
# ==============================================================================
# STEP 4: SCHEMA CONTRACT VALIDATION (Phase 6): names AND types
# ==============================================================================
 
def schema_to_contract(schema: "StructType", title: str, description: str) -> Dict[str, Any]:
    """Serialises a *_SCHEMA into the JSON contract format used in data/schemas/."""
    return {"title": title, "description": description, "type": "struct",
            "fields": [{"name": f.name, "type": f.dataType.typeName(), "nullable": f.nullable,
                        "metadata": dict(f.metadata or {})} for f in schema.fields]}
 
 
def validate_silver_schema_contract(df: "DataFrame", expected_schema_json_path: str,
                                    check_nullability: bool = False) -> Tuple[bool, List[str]]:
    """Checks presence, no extras and exact data types against the authoritative JSON contract.
    Nullability is opt-in: computed DataFrame columns are always nullable in Spark; NOT NULL is enforced by the
    Delta table schema (ensure_delta_table) at write time. Use check_nullability=True on a table read back from disk."""
    with open(expected_schema_json_path, "r", encoding="utf-8") as fh:
        contract = json.load(fh)
    expected = {x["name"]: x for x in contract["fields"]}
    actual = {f.name: f for f in df.schema.fields}
    errors: List[str] = []
    for name, spec in expected.items():
        if name not in actual:
            errors.append(f"Missing column: '{name}' (expected {spec['type']})")
            continue
        if actual[name].dataType.typeName().lower() != str(spec["type"]).lower():
            errors.append(f"Type mismatch on '{name}': actual '{actual[name].dataType.typeName()}' != expected '{spec['type']}'")
        if check_nullability and not spec.get("nullable", True) and actual[name].nullable:
            errors.append(f"Nullability mismatch on '{name}': contract says NOT NULL")
    errors += [f"Unexpected extra column: '{n}'" for n in actual if n not in expected]
    return (len(errors) == 0), errors
 

In [0]:

# ==============================================================================
# STEP 5: DELTA PERSISTENCE (partition-pruned, zero write amplification)
# ==============================================================================
 
def ensure_delta_table(spark, path: str, schema: "StructType", table_name: Optional[str] = None) -> None:
    """Creates the empty, explicitly typed, date-partitioned Delta table on first run."""
    if not DeltaTable.isDeltaTable(spark, path):
        print(f"[Silver Persistence] Creating Delta table at {path}")
        spark.createDataFrame([], schema).write.format("delta").partitionBy("measurement_date").save(path)
    if table_name:
        try:
            spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{path}'")
        except Exception:
            pass
 
 
def _window_predicate(window_start: Optional[str], window_end: Optional[str]) -> str:
    if window_start and window_end:
        return (f" AND target.measurement_date >= '{window_start}' AND target.measurement_date <= '{window_end}'")
    return ""
 
 
def _merge_metrics(target: "DeltaTable") -> Dict[str, int]:
    try:
        m = target.history(1).select("operationMetrics").first()["operationMetrics"]
        return {"num_target_rows_inserted": int(m.get("numTargetRowsInserted", 0)),
                "num_target_rows_updated": int(m.get("numTargetRowsUpdated", 0))}
    except Exception as err:
        print(f"[Silver Persistence Warning] Could not read MERGE metrics: {err}")
        return {"num_target_rows_inserted": 0, "num_target_rows_updated": 0}
 
 
def merge_silver_network_measurements(spark, df_silver: "DataFrame", silver_path: str,
                                      window_start: Optional[str], window_end: Optional[str]) -> Dict[str, int]:
    """
    ON measurement_id AND measurement_date (+ literal window) -> partition pruning.
    WHEN MATCHED only if a business attribute actually changed (null-safe <=>); _batch_id never triggers it,
    so re-running identical data yields 0 inserts / 0 updates.
    """
    ensure_delta_table(spark, silver_path, SILVER_SCHEMA, _table_name("silver", "silver_network_measurements"))
    target = DeltaTable.forPath(spark, silver_path)
    cond = ("target.measurement_id = source.measurement_id AND target.measurement_date = source.measurement_date"
            + _window_predicate(window_start, window_end))
    changed = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in PARENT_MERGE_COMPARE_COLUMNS)
    update_set = {c: f"source.{c}" for c in PARENT_MERGE_COMPARE_COLUMNS}
    update_set.update({"load_timestamp": "source.load_timestamp", "_batch_id": "source._batch_id"})
    (target.alias("target").merge(df_silver.alias("source"), cond)
        .whenMatchedUpdate(condition=changed, set=update_set)
        .whenNotMatchedInsertAll().execute())
    metrics = _merge_metrics(target)
    print(f"[Silver Persistence] silver_network_measurements: {metrics}")
    return metrics
 
 
def merge_child_delta_table(spark, df_child: "DataFrame", child_path: str, schema: "StructType", key_column: str,
                            window_start: Optional[str], window_end: Optional[str],
                            table_name: Optional[str] = None) -> Dict[str, int]:
    """Insert-only, partition-pruned merge for the immutable child projections."""
    ensure_delta_table(spark, child_path, schema, table_name)
    target = DeltaTable.forPath(spark, child_path)
    cond = (f"target.{key_column} = source.{key_column} AND target.measurement_date = source.measurement_date"
            + _window_predicate(window_start, window_end))
    target.alias("target").merge(df_child.alias("source"), cond).whenNotMatchedInsertAll().execute()
    metrics = _merge_metrics(target)
    print(f"[Silver Persistence] {child_path.rsplit('/', 1)[-1]}: {metrics}")
    return metrics
 
 
def quarantine_invalid_rows(spark, df_invalid: "DataFrame", batch_id: str, quarantine_path: str) -> int:
    """Idempotent: MERGE on (raw_record, _source_file) so re-runs never duplicate quarantine rows."""
    n = df_invalid.count()
    if n == 0:
        return 0
    staged = (df_invalid.select(
        F.col("raw_record"), F.lit(QUARANTINE_SOURCE_TAG).alias("_source_file"), F.lit(batch_id).alias("_batch_id"),
        F.current_timestamp().alias("quarantine_timestamp"),
        F.concat(F.lit("Silver DQ violation: "), F.col("_dq_reason")).alias("quarantine_reason")).dropDuplicates(["raw_record"]))
    if DeltaTable.isDeltaTable(spark, quarantine_path):
        (DeltaTable.forPath(spark, quarantine_path).alias("t")
            .merge(staged.alias("s"), "t.raw_record = s.raw_record AND t._source_file = s._source_file")
            .whenNotMatchedInsertAll().execute())
    else:
        staged.write.format("delta").mode("append").save(quarantine_path)
    print(f"[Quarantine Router] {n} rows violated Silver data-quality rules -> {quarantine_path}")
    return n
 
 
def optimize_silver(spark, silver_path: str, window_start: Optional[str], window_end: Optional[str]) -> None:
    """ZORDER (probe_cc, probe_asn, tampering_vector) on the processed partitions only; never fails the run."""
    try:
        opt = DeltaTable.forPath(spark, silver_path).optimize()
        if window_start and window_end:
            opt = opt.where(f"measurement_date >= '{window_start}' AND measurement_date <= '{window_end}'")
        opt.executeZOrderBy("probe_cc", "probe_asn", "tampering_vector")
    except Exception as err:
        print(f"[Silver Optimize Warning] OPTIMIZE/ZORDER skipped: {err}")
 

In [0]:

# ==============================================================================
# STEP 6: ORCHESTRATION + AUDIT LOGGING
# ==============================================================================
 
class _NullAuditor:
    def __enter__(self): return self
    def __exit__(self, *exc): return False
    def set_metrics(self, **kw): pass
 
 
def _make_auditor(session, parameter: str, load_type: str):
    if audit_logger and hasattr(audit_logger, "PipelineAuditor"):
        return audit_logger.PipelineAuditor(layer="Bronze-to-Silver", parameter=parameter,
                                            load_type=load_type, spark=session)
    return _NullAuditor()
 
 
def run_bronze_to_silver(
    spark: Optional[Any] = None, batch_date: Optional[str] = None, load_type: Optional[str] = None,
    lookback_days: Optional[int] = None, backfill_start: Optional[str] = None, backfill_end: Optional[str] = None,
    country_filter: Optional[str] = None, bronze_path: Optional[str] = None, silver_path: Optional[str] = None,
    quarantine_path: Optional[str] = None, run_optimize: bool = True, write_children: bool = True,
    pepper_salt: Optional[str] = None, allow_dev_salt: bool = False,
) -> Dict[str, Any]:
    """
    Bronze -> Silver for any date window; safe to re-run.
      incremental: run_bronze_to_silver(batch_date="2026-09-28")
      backfill   : run_bronze_to_silver(load_type="backfill", backfill_start="2026-09-10", backfill_end="2026-09-14")
      one day    : run_bronze_to_silver(load_type="backfill", batch_date="2026-09-22", lookback_days=0)
      full       : run_bronze_to_silver(load_type="full")
    """
    p = resolve_silver_parameters(batch_date, load_type, lookback_days, backfill_start, backfill_end, country_filter)
    pepper = resolve_pepper_salt(pepper_salt, allow_dev_fallback=allow_dev_salt)   # fail-closed BEFORE any read/write
    session = resolve_spark_session(spark)
    bronze_path = bronze_path or config.PATHS["bronze"]
    silver_path = silver_path or config.PATHS["silver"]
    quarantine_path = quarantine_path or config.PATHS["quarantine"]
    dns_path, tcp_path = _table_path("silver_dns", "silver_dns_experiments"), _table_path("silver_tcp", "silver_tcp_experiments")
    app_path = _table_path("silver_app", "silver_app_anomaly_measurements")
    load_ts = datetime.now(timezone.utc).replace(tzinfo=None)
 
    summary: Dict[str, Any] = dict(p)
    with _make_auditor(session, p["batch_id"], p["load_type"]) as aud:
        print(f"[Silver] batch_id={p['batch_id']} window=[{p['window_start']} .. {p['window_end']}] country={p['country_filter']}")
        df_raw = read_bronze_window(session, bronze_path, p["window_start"], p["window_end"], p["country_filter"])
        df_in, drift = detect_and_absorb_schema_drift(df_raw)
        rows_read = df_in.count()
 
        out = transform_bronze_to_silver(df_in, read_taxonomy(session), pepper, p["batch_id"], load_ts)
        quarantined = quarantine_invalid_rows(session, out["invalid"], p["batch_id"], quarantine_path)
        parent = out["parent"]
        rows_staged = parent.count()
        dup_dropped = max(rows_read - quarantined - rows_staged, 0)
        if dup_dropped:
            print(f"[Silver] {dup_dropped} duplicate measurement_id rows collapsed before MERGE.")
 
        pm = (merge_silver_network_measurements(session, parent, silver_path, p["window_start"], p["window_end"])
              if rows_staged > 0 else {"num_target_rows_inserted": 0, "num_target_rows_updated": 0})
        aud.set_metrics(rows_read=rows_read, rows_inserted=pm["num_target_rows_inserted"],
                        rows_updated=pm["num_target_rows_updated"], rows_quarantined=quarantined)
 
    child_metrics: Dict[str, Dict[str, int]] = {}
    if write_children and rows_staged > 0:
        for name, key, df_child, path, schema, tkey in (
            ("silver_dns_experiments", "query_id", out["dns"], dns_path, SILVER_DNS_SCHEMA, "silver_dns"),
            ("silver_tcp_experiments", "tcp_connect_id", out["tcp"], tcp_path, SILVER_TCP_SCHEMA, "silver_tcp"),
            ("silver_app_anomaly_measurements", "app_measurement_id", out["app"], app_path, SILVER_APP_SCHEMA, "silver_app"),
        ):
            with _make_auditor(session, f"{p['batch_id']}|{name}", p["load_type"]) as caud:
                n_child = df_child.count()
                cm = (merge_child_delta_table(session, df_child, path, schema, key, p["window_start"], p["window_end"],
                                              _table_name(tkey, name)) if n_child > 0
                      else {"num_target_rows_inserted": 0, "num_target_rows_updated": 0})
                caud.set_metrics(rows_read=n_child, rows_inserted=cm["num_target_rows_inserted"],
                                 rows_updated=0, rows_quarantined=0)
                child_metrics[name] = cm
    if run_optimize and rows_staged > 0:
        optimize_silver(session, silver_path, p["window_start"], p["window_end"])
 
    summary.update({"rows_read": rows_read, "rows_staged": rows_staged, "duplicates_collapsed": dup_dropped,
                    "rows_quarantined": quarantined, "rows_inserted": pm["num_target_rows_inserted"],
                    "rows_updated": pm["num_target_rows_updated"], "child_tables": child_metrics,
                    "schema_drift": drift, "silver_path": silver_path,
                    "status": "Success" if quarantined == 0 else "Partial_Quarantine"})
    return summary
 

In [0]:
   import importlib
   cfg = importlib.import_module("00_config")
   print(cfg.PATHS)
   print(cfg.TABLE_NAMES)

In [0]:

if __name__ == "__main__":
    print("=" * 75)
    print(" PROJECT PANOPTICON: SILVER TRANSFORMATION ENGINE (phases 1-6)")
    print("=" * 75)
    init_silver_widgets()
    result = run_bronze_to_silver()
    for k, v in result.items():
        print(f"  * {k:<20}: {v}")
 